# Quick check — Task 2: fixation-probability prediction

Runs the fixation-probability pipeline (`src/fixation_prediction/`) end to end on a synthetic population with known selection coefficients:

1. build the data matrix from a synthetic FASTA with `FluStudy`;
2. for one prediction date, infer MPL selection coefficients, label each mutation's future fate, and estimate P(fixation) / P(loss) with forward Wright–Fisher simulations;
3. repeat for a few dates and compute the AUROC / fixation-probability summary.

Runtime: about 2–3 minutes, plus compilation on the first run. The production settings are in `src/fixation_prediction/run_fixation_prediction.jl`.

In [ ]:
# Run from the notebooks/ folder (or the project root)
const PROJECT_ROOT = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
using Pkg; Pkg.activate(PROJECT_ROOT)
using Dates, DataFrames, Statistics, Random

include(joinpath(PROJECT_ROOT, "src", "fixation_prediction", "FixationPrediction.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "FluStudy.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "synthetic_data.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "cli.jl"))                    # run_name (output file names)
using .FixationPrediction, .SyntheticFlu
const AA = "-ACDEFGHIKLMNPQRSTVWY"                   # integer state -> amino-acid letter

## 1. Synthetic input

`simulate_population` (`src/shared/synthetic_data.jl`) evolves a population with the negative-binomial dynamics of `src/shared/population_simulation.jl`, using its default parameters:

* 20 sites with 4 states each; selection coefficients `s ~ Normal(0, 0.05)`, with 0 for the alleles of a random reference sequence;
* 100 random starting sequences sharing 1000 individuals;
* offspring numbers negative binomial with R = 2 and dispersion k = 0.1, population ceiling 10 000;
* mutation rate 10⁻³ per site per generation;
* 1000 generations, one generation per day (2009-01-02 … 2011-09-28).

Each day 20 individuals are sampled and "sequenced" (states 1–4 are written as the amino acids A, C, D, E). The samples are saved as a FASTA file with the same header layout as the real data, then read back with `FluStudy.create_study`, as the command-line scripts do (`protein_length = 20` instead of 566 for HA).

In [ ]:
syn = simulate_population(seed = 1, n_sample = 20)
fasta = joinpath(mktempdir(), "synthetic_H3N2_HA.fasta")
write_fasta(fasta, syn)

start_date, stop_date = Date(2009, 1, 1), Date(2009, 1, 1) + Day(1000)
study = redirect_stdout(devnull) do
    FluStudy.create_study(fasta; protein = "HA", protein_length = syn.L,
                          start_date = start_date, stop_date = stop_date, tstep = 10.0, tsmooth = 60.0)
end
data = study.data
println("population size: median ", round(Int, median(syn.pop_size)), ", range ", extrema(syn.pop_size),
        " | distinct sequences in the population: median ", round(Int, median(syn.n_species)))
println("selection coefficients: ", count(>(0.05), syn.s), " alleles with s > 0.05, ", count(<(-0.05), syn.s), " with s < -0.05 (of ", length(syn.s), ")")

In [ ]:
# FluStudy reproduces the synthetic samples: the same (day, count, sequence) rows, up to their order
# within a day. It drops the last FASTA date, because its daily bins are right-open.
sorted_rows(m) = sort(Tuple.(eachrow(m)))
@assert sorted_rows(data) == sorted_rows(syn.data[syn.data[:, 1] .< maximum(syn.data[:, 1]), :])
println("data matrix: ", size(data), "  (rows × [day, count, $(syn.L) sites])")

## 2. One prediction date

`run_simulation` keeps the rows between `start_date` and `stop_date`, takes the observed day closest to the prediction date, fits the MPL selection coefficients on the data up to that day, labels each mutation's future fate (`fixed` / `lost` / `poly`) from the full trajectory, and runs `M` forward simulations (`M = 1000` in production, 15 here).

The true coefficients are shown relative to the pipeline's reference sequence (the consensus at the first time point), the same normalisation as the inferred ones.

In [ ]:
Random.seed!(1)
prediction_date = Date(2010, 5, 16)              # day 500
tr = redirect_stdout(devnull) do                 # silence the per-trajectory log
    run_simulation(data, prediction_date; start_date = start_date, stop_date = stop_date, M = 15)
end

reference = [tr[(i - 1) * 21 + 1].reference for i in 1:syn.L]
s_true = normalize_to_reference(syn.s21, reference)
allele_index(x) = (x.i - 1) * 21 + x.val

# Mutations at intermediate frequency at the prediction date
active = sort(filter(x -> x.val != x.reference && 0.05 < x.freq[end] < 0.95, tr), by = x -> -x.freq[end])
DataFrame(site      = getfield.(active, :i),
          mutation  = [string(AA[x.reference], x.i, AA[x.val]) for x in active],
          freq      = round.([x.freq[end] for x in active], digits = 3),
          s_true    = round.([s_true[allele_index(x)] for x in active], digits = 3),
          sc_MPL    = round.(getfield.(active, :sc), digits = 3),
          P_fix     = getfield.(active, :porcent_simulation_fix),
          P_loss    = getfield.(active, :porcent_simulation_lost),
          true_fate = getfield.(active, :fixation))

In [ ]:
# Sanity checks
@assert length(tr) == syn.L * 21                               # one entry per (site, state)
@assert all(0 .<= getfield.(tr, :porcent_simulation_fix) .<= 1)
@assert all(0 .<= getfield.(tr, :porcent_simulation_lost) .<= 1)
@assert all(x -> x.fixation in (:fixed, :lost, :poly), tr)
@assert all(x -> x.T0 == 500, tr)                              # observed day closest to prediction_date

fixed = filter(x -> x.fixation == :fixed, active)
lost  = filter(x -> x.fixation == :lost, active)
println("mean P(fix): mutations that later fixed = ", round(mean(getfield.(fixed, :porcent_simulation_fix)), digits = 2),
        " (n = $(length(fixed))), mutations later lost = ", round(mean(getfield.(lost, :porcent_simulation_fix)), digits = 2),
        " (n = $(length(lost)))")

## 3. Several dates and the summary curves

Each date writes one CSV named as `run_fixation_prediction.jl` names them (`fixation_<subtype>_<protein>_<date>_noise<n>_tsmooth<σ>_gamma<γ>.csv`). `load_results` reads them all, and `compute_summary_curves` bins the mutations by current frequency. Per bin it reports the AUROC of the simulation score P(fix) − P(loss) against the true fixed/lost labels, the AUROC of the null model (current frequency), and the fixation probability of the top and bottom 50% of predictions. With only four dates and `M = 5`, many bins are empty (`NaN`); the production runs use about 400 dates.

In [ ]:
dates = [Date(2009, 10, 1), Date(2010, 3, 1), Date(2010, 8, 1), Date(2011, 1, 1)]
results_dir = mktempdir()
redirect_stdout(devnull) do
    for (j, d) in enumerate(dates)
        Random.seed!(j)
        run_simulation(data, d; start_date = start_date, stop_date = stop_date, M = 5,
                       out_file = joinpath(results_dir, run_name("fixation", "H3N2", "HA", d; noise = 1, tsmooth = 60.0, gamma = 20.0) * ".csv"))
    end
end
df_all = load_results(results_dir)
freqs, auc_sim, auc_null, fix_top, fix_bottom, fix_all = compute_summary_curves(df_all)
DataFrame(frequency = freqs, AUROC_sim = auc_sim, AUROC_null = auc_null,
          Pfix_top50 = fix_top, Pfix_bottom50 = fix_bottom, Pfix_all = fix_all)

In [ ]:
@assert nrow(df_all) == length(dates) * syn.L * 21
@assert all(x -> isnan(x) || 0 <= x <= 1, [auc_sim; auc_null; fix_top; fix_bottom; fix_all])
println("dates: ", length(dates), ", rows (site × state × date): ", nrow(df_all))